# Serving — Google rating per day (agg_restaurant_rating_daily)

Since 2026-11-06 this notebook only builds the serving table. The Gold fact `fact_place_rating_snapshot_poc` (place × read) is built by `gold/fact_place_rating_snapshot_poc` in the job "2. gold (poc tables)"; the old `fact_place_rating_snapshot` is no longer written.

| Table | Layer | Grain | Load |
| --- | --- | --- | --- |
| `agg_restaurant_rating_daily` | Serving | place × Vietnam day, **dense** from the first read to today (last read carried forward) | full rebuild |

`rating`, `rating_count` of the latest read on or before the day (semi-additive), `rating_change` / `rating_count_gained` vs the previous day (non-zero only on a day with a new read; additive over days, NULL on the first day), `is_new_read`, `days_since_read`, `read_at`, `read_source`. `restaurant_id` = current `dim_restaurant_scd2` row with that `place_id` (`'unknown'` if none).


In [ ]:
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))
dbutils.widgets.text("serving", "prod.serving")
SERVING = dbutils.widgets.get("serving")  # serving layer: agg_* tables read by the dashboard / app / chat (2026-11-02)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SERVING}")

SNAPSHOT_TZ = "Asia/Ho_Chi_Minh"
FACT = f"{GOLD}.fact_place_rating_snapshot_poc"
DAILY = f"{SERVING}.agg_restaurant_rating_daily"
print(f"Gold: {FACT} · Serving: {DAILY}")

In [ ]:
TODAY = f"to_date(from_utc_timestamp(current_timestamp(), '{SNAPSHOT_TZ}'))"
spark.sql(f"""
    CREATE OR REPLACE TABLE {DAILY}
    COMMENT 'Serving: 1 Google place (restaurant) x 1 Vietnam day, dense (last read carried forward), from fact_place_rating_snapshot_poc. rating / rating_count semi-additive, rating_change / rating_count_gained additive over days.'
    AS
    WITH day_reads AS (
        SELECT place_id, to_date(from_utc_timestamp(snapshot_ts, '{SNAPSHOT_TZ}')) AS read_date,
               max_by(named_struct('rating', rating, 'rating_count', rating_count, 'read_at', snapshot_ts,
                                   'source', source), snapshot_ts) AS r
        FROM {FACT}
        GROUP BY place_id, to_date(from_utc_timestamp(snapshot_ts, '{SNAPSHOT_TZ}'))
    ),
    days AS (
        SELECT place_id, explode(sequence(MIN(read_date), {TODAY})) AS snapshot_date
        FROM day_reads GROUP BY place_id
    ),
    filled AS (
        SELECT d.place_id, d.snapshot_date, r.r IS NOT NULL AS is_new_read,
               last_value(r.r, true) OVER (PARTITION BY d.place_id ORDER BY d.snapshot_date
                                           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS r
        FROM days d
        LEFT JOIN day_reads r ON r.place_id = d.place_id AND r.read_date = d.snapshot_date
    ),
    restaurant AS (
        SELECT place_id, min(restaurant_id) AS restaurant_id
        FROM {GOLD}.dim_restaurant_scd2 WHERE current_flag = 'Y' AND restaurant_id <> 'unknown' AND place_id IS NOT NULL GROUP BY place_id
    )
    SELECT
        CAST(date_format(f.snapshot_date, 'yyyyMMdd') AS INT)              AS snapshot_date_key,
        COALESCE(dr.restaurant_id, 'unknown')                              AS restaurant_id,
        f.place_id,
        f.r.rating                                                         AS rating,
        f.r.rating_count                                                   AS rating_count,
        ROUND(f.r.rating - LAG(f.r.rating) OVER w, 2)                      AS rating_change,
        f.r.rating_count - LAG(f.r.rating_count) OVER w                    AS rating_count_gained,
        f.is_new_read,
        LAG(f.snapshot_date) OVER w IS NULL                                AS is_first_snapshot,
        datediff(f.snapshot_date, to_date(from_utc_timestamp(f.r.read_at, '{SNAPSHOT_TZ}'))) AS days_since_read,
        f.r.read_at                                                        AS read_at,
        f.r.source                                                         AS read_source,
        current_timestamp()                                                AS etl_date
    FROM filled f
    LEFT JOIN restaurant dr ON dr.place_id = f.place_id
    WINDOW w AS (PARTITION BY f.place_id ORDER BY f.snapshot_date)
""")

In [ ]:
s = spark.sql(f"""
    SELECT COUNT(*) AS n, COUNT(DISTINCT place_id, snapshot_date_key) AS grain, COUNT(DISTINCT place_id) AS places,
           COUNT_IF(restaurant_id = 'unknown') AS unknown_restaurant, MAX(snapshot_date_key) AS last_day,
           COUNT_IF(rating_change <> 0) AS rating_changes
    FROM {DAILY}
""").collect()[0]
print(f"📊 {DAILY}: {s['n']} rows · {s['places']} places · last day {s['last_day']} · {s['rating_changes']} rating changes · "
      f"{s['unknown_restaurant']} rows without a restaurant (unknown) · grain (place_id x day): " + ("✅" if s["n"] == s["grain"] else "❌ duplicates"))
if s["n"] != s["grain"]:
    raise AssertionError(f"{DAILY}: duplicate (place_id, day) rows")
display(spark.sql(f"""
    SELECT place_id, snapshot_ts, source, rating, rating_change, rating_count, rating_count_gained
    FROM {FACT} WHERE NOT is_first_snapshot ORDER BY abs(rating_change) DESC, rating_count_gained DESC LIMIT 20
"""))
print("\n--- place_rating complete ---")
spark.sql(f"DROP TABLE IF EXISTS {GOLD}.agg_restaurant_rating_daily")  # moved to {SERVING} on 2026-11-02